# Анализ рынка видеоигр: поиск закономерностей успеха

У нас есть массив исторических данных об индустрии видеоигр: сведения о продажах, оценки критиков и пользователей, жанровая принадлежность и игровые платформы (Xbox, PlayStation и другие). Задача — найти закономерности, которые определяют успех игры на рынке. Это даст возможность делать ставку на перспективные продукты и грамотно выстраивать рекламные кампании.

Данные охватывают период вплоть до 2016 года. Представим, что на дворе декабрь 2016-го, а мы готовим кампанию на 2017 год — именно с этой позиции и будем смотреть на цифры.

В датасете встречается аббревиатура ESRB (Entertainment Software Rating Board) — это организация, которая присваивает компьютерным играм возрастной рейтинг. Она анализирует игровой контент и относит его к подходящей категории: например, «Для взрослых», «Для детей младшего возраста» или «Для подростков».

Как будем работать
- Познакомимся с данными и оценим их общую структуру
- Проведём предварительную обработку и подготовку
- Выполним исследовательский анализ и найдём ключевые закономерности
- Составим портрет типичного пользователя для каждого региона
- Проверим статистические гипотезы
- Соберём всё воедино и сформулируем общие выводы

### Откроем файл и изучим общую информацию

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st

In [13]:
try:
    data = pd.read_csv('./games.csv')
except:
    print('Ошибка при загрузке датасета', )
else:
    print('Датасет загружен успешно')

Датасет загружен успешно


Выведем первые 10 строк на экран: 

In [14]:
data.head(10)

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.0,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
6,New Super Mario Bros.,DS,2006.0,Platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,Misc,13.96,9.18,2.93,2.84,58.0,6.6,E
8,New Super Mario Bros. Wii,Wii,2009.0,Platform,14.44,6.94,4.70,2.24,87.0,8.4,E
9,Duck Hunt,NES,1984.0,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN


Выведем общую информацию о датасете:

In [15]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 1.4 MB


Проверим данные на наличие явных дубликатов:

In [16]:
print(f'Количество дубликатов: {data.duplicated().sum()}')

Количество дубликатов: 0


После знакомства с данными становятся заметны моменты, которые требуют внимания:
- Названия столбцов стоит привести к нижнему регистру — так с ними удобнее работать
- Часть столбцов нужно привести к корректным типам данных
- В некоторых столбцах встречаются пропуски (NaN)
- Имеет смысл добавить новый столбец — суммарные продажи игры по всем регионам

### Подготовка данных

Заменим названия столбцов, приведя их к нижнему регистру:

In [17]:
data.columns = data.columns.str.lower()

In [18]:
data.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Приведём столбцы с годом выпуска и оценкой критиков к целочисленному типу, так как эти признаки могут принимать только целые значения:

In [19]:
data['year_of_release'] = data['year_of_release'].astype(dtype='Int64').copy()
data['critic_score'] = data['critic_score'].astype(dtype='Int64').copy()

Теперь разберёмся со столбцом user_score. Здесь есть ~2400 строк, в которых значение оценки пользователей заполнено строкой "tbd", что означает "to be decided", то есть оценка подлежит уточнению. Такие значения можно считать пропусками и заполнить их как Nan, чтобы у нас была возможность привести этот столбец к типу float:

In [23]:
data['user_score'].replace('tbd', np.nan, inplace=True)

C:\Users\Дом\AppData\Local\Temp\ipykernel_3892\1947917369.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['user_score'].replace('tbd', np.nan, inplace=True)


0          8
1        NaN
2        8.3
3          8
4        NaN
        ... 
16710    NaN
16711    NaN
16712    NaN
16713    NaN
16714    NaN
Name: user_score, Length: 16715, dtype: str

А столбец с оценкой пользователей приведём из типа object в тип float, чтобы у нас была возможность работать с оценкой пользователей как с числом:

In [25]:
data['user_score'] = pd.to_numeric(data['user_score'], errors='coerce')

Теперь посмотрим на количество пропусков:

In [26]:
data.isna().sum()

name                  2
platform              0
year_of_release     269
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8578
user_score         9125
rating             6766
dtype: int64

In [27]:
data[data['name'].isna()].head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
659,NaN,GEN,1993,NaN,1.78,0.53,0.00,0.08,<NA>,NaN,NaN
14244,NaN,GEN,1993,NaN,0.00,0.00,0.03,0.00,<NA>,NaN,NaN


In [28]:
data.isna().sum()

name                  2
platform              0
year_of_release     269
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8578
user_score         9125
rating             6766
dtype: int64

Теперь перейдём к столбцу "rating". Возможно, эти пропуски появились вследствие того, что ассоциация ESRB не определила возрастную категорию для данной игры. Такие пропуски можно заменить на значение "Unknown", чтобы было удобнее группировать такие игры без категории в дальнейшем.

In [29]:
data['rating'].fillna(value='Unknown', inplace=True)

C:\Users\Дом\AppData\Local\Temp\ipykernel_3892\126589215.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['rating'].fillna(value='Unknown', inplace=True)


0              E
1        Unknown
2              E
3              E
4        Unknown
          ...   
16710    Unknown
16711    Unknown
16712    Unknown
16713    Unknown
16714    Unknown
Name: rating, Length: 16715, dtype: str

In [30]:
data['total_sales'] = data['na_sales'] + data['eu_sales'] + data['jp_sales'] + data['other_sales']